In [1]:
import cv2
import numpy as np
from pyzbar import pyzbar
from PIL import Image, ImageDraw
import pandas as pd
from datetime import datetime

1. camera initialization & Frame reading
2. Preprocessing of captured frame
3. QR code reading 
4. Excel file making with 1 column of read UID
5. Make Main_Loop() for continuous function

In [2]:
def cam_init(dev):
    cam=cv2.VideoCapture(dev)
    cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    return cam

In [3]:
def preprocess_frame(frame):
    gray_frame=cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    eq_frame=cv2.equalizeHist(gray_frame)
    threshold_val=cv2.adaptiveThreshold(
        eq_frame, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=51,
        C=10
    )
    return cv2.cvtColor(threshold_val, cv2.COLOR_GRAY2BGR)

In [4]:
def scan_qr(frame):
    return pyzbar.decode(frame)

In [5]:
def draw_bounding(image_pil, barcode):
    draw_bounding = ImageDraw.Draw(image_pil)

    rect = barcode.rect  # <-- fix typo here
    draw_bounding.rectangle(
        [(rect.left, rect.top),
         (rect.left + rect.width, rect.top + rect.height)],
        outline="#0b15db", width=3
    )
    polygon_box = [(pt.x, pt.y) for pt in barcode.polygon]
    draw_bounding.polygon(polygon_box, outline="#e945ff")

In [6]:
def save_records_to_csv(records, filename="uav_scan_report.xlsx"):
    df = pd.DataFrame(records)
    df.to_excel(filename, index=False)
    print(f"[+] Saved {len(df)} scan records to {filename}")

In [ ]:
def main_loop():
    cam = cam_init(dev="https://192.168.1.41:8080/video")
    seen_ids = set()
    records = []
    print("Starting enhanced QR‑scan session. Press 'q' to quit.")

    while True:
        ret, frame = cam.read()
        if not ret:
            continue

        process = preprocess_frame(frame)
        decoded_orig = scan_qr(frame)
        decoded_proc = scan_qr(process)
        decoded = {d.data: d for d in (decoded_orig + decoded_proc)}.values()
        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        for obj in decoded:
            model_id = obj.data.decode('utf-8')
            if model_id not in seen_ids:
                seen_ids.add(model_id)
                ts = datetime.now().isoformat(timespec='seconds')
                records.append({'timestamp': ts, 'model_id': model_id})
                print(f"[QR] {model_id} @ {ts}")

            draw_bounding(pil_img, obj)

        disp = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        cv2.imshow("Enhanced QR Scanner", disp)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cam.release()
    cv2.destroyAllWindows()
    save_records_to_csv(records)

In [8]:
if __name__ == "__main__":
    main_loop()

[*] Starting enhanced QR‑scan session. Press 'q' to quit.
[QR] 67578985 @ 2025-07-09T19:49:20
[QR] https://www.bing.com @ 2025-07-09T19:50:09


KeyboardInterrupt: 